# Vietnamese-History Trajectory Dataset — V4.4 Fast GPU

Tracked Colab source with pooled Agent-FLAN full normalization.


## Cell 1 — GPU/XLA guards

Phải chạy đầu tiên để tránh JAX/XLA giữ VRAM. Không import model trước cell này.

In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.05"
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("GPU/XLA guards configured.")

## Cell 2 — Config

Đây là cell bạn chỉnh trước khi chạy.

### Preset chất lượng giữ nguyên

Các budget đã pass smoke trước vẫn giữ nguyên:

- `TOP_K = 6`
- `OBSERVATION_CHAR_BUDGET = 3200`
- `TRAJECTORY_OBSERVATION_CHAR_BUDGET = 5200`
- `MAX_RESULT_TEXT_CHARS = 650`
- `MAX_SEQ_LENGTH = 4096`

### Performance preset cho retrieval/rerank

Notebook này chỉ tăng tốc ở các knob **không làm thay đổi schema hay budget evidence**:

- tự nhận GPU và chọn `RERANK_BATCH_SIZE` phù hợp;
- tăng `CHECKPOINT_EVERY` để giảm số lần ghi checkpoint lên Google Drive;
- giữ `TOP_K`, observation budget, corpus/index và logic builder nguyên vẹn.

Khuyến nghị GPU cho Cell 15/23:

| GPU | VRAM | RERANK_BATCH_SIZE auto | Khuyến nghị |
|---|---:|---:|---|
| T4 | 16 GB | 8 | Chạy được, nhưng chậm hơn |
| L4 / A10 | ~24 GB | 16 | **Khuyến nghị / cân bằng tốt** |
| A100 40 GB | 40 GB | 24 | Nhanh hơn nếu có sẵn |
| A100/H100 80 GB | 80 GB | 32 | Rất nhanh nhưng thường dư cho job này |

`AUTO_TUNE_RERANK_BATCH=True` là mặc định. Nếu gặp CUDA OOM, tắt auto và hạ batch lần lượt `16 → 8 → 4`.

> Lưu ý: tăng rerank batch chỉ tăng throughput của cross-encoder reranker. Builder vẫn tạo trajectory theo từng candidate/query, nên không kỳ vọng tốc độ tăng tuyến tính theo batch.

Teacher vẫn OFF. `TEACHER_BATCH_SIZE` không liên quan tới full deterministic build hiện tại.

`APPROVE_FULL_BUILD=False` để notebook không vô tình chạy full trước khi smoke mới được review.

Bản performance dùng `RUN_TAG="fast"` để không resume nhầm output cũ được tạo với batch/config khác.


In [ ]:
REPO_URL = "https://github.com/ChuTungDuongg/Chatbot_answering_vietnamese_history.git"
BRANCH = "main"
REPO_DIR = "/content/Chatbot_answering_vietnamese_history"

DRIVE_MOUNT = "/content/drive"
DRIVE_ROOT = "/content/drive/MyDrive/vn-history"
DEPLOYMENT_ROOT = f"{DRIVE_ROOT}/artifacts/vn_history_deployment"
CORPUS_PATH = f"{DEPLOYMENT_ROOT}/corpus"
HF_CACHE_DIR = f"{DRIVE_ROOT}/hf_cache"

BASE_RUN_NAME = "trajectory_dataset_v4_3"
RUN_TAG = "fast"  # tránh resume nhầm run cũ khi đổi performance config

FINAL_MAX_SAMPLES = 4000
MIN_ACCEPTABLE_MIXED_ROWS = 3800

AGENT_FLAN_MAX = 700
MULTIHOP_MAX = 900
VIETNAM_HISTORY_MAX = 900

CUSTOM_COUNTS = {
    "factual": 500,
    "cause": 350,
    "significance": 250,
    "compare": 250,
    "summary": 250,
    "multihop": 200,
    "verification": 150,
    "hard_negative": 150,
    "insufficient_evidence": 100,
}
assert sum(CUSTOM_COUNTS.values()) == 2200

MIX_RATIOS = {
    "custom_history": 0.55,
    "multi_hop_function_calling": 0.17,
    "agent_flan": 0.12,
    "vietnam_history_200k": 0.16,
}
assert abs(sum(MIX_RATIOS.values()) - 1.0) < 1e-9

TOP_K = 6

# Performance: Cell 7 sẽ tự resolve batch theo VRAM trước khi Build Context được ghi.
AUTO_TUNE_RERANK_BATCH = True
RERANK_BATCH_SIZE = 4          # fallback / manual value khi AUTO_TUNE=False
RERANK_BATCH_SIZE_CAP = 32     # giới hạn an toàn cho notebook này

MAX_CORPUS_RECORDS = 10_000

# Conservative 4096-token defaults after the real smoke failure.
OBSERVATION_CHAR_BUDGET = 3_200
TRAJECTORY_OBSERVATION_CHAR_BUDGET = 5_200
MAX_RESULT_TEXT_CHARS = 650
MAX_SEQ_LENGTH = 4096

SEED = 42
# Ít checkpoint hơn => giảm Google Drive I/O trong full custom 2.200.
# Nếu runtime rất hay disconnect, có thể hạ lại 50.
CHECKPOINT_EVERY = 100
PUBLIC_CHECKPOINT_EVERY = 100
HISTORY_MODE = "style_only"

USE_GPU_FOR_RETRIEVAL = True
MIN_FREE_GPU_MB = 6000

# Optional stronger smoke after smoke-30.
RUN_EXTENDED_SMOKE_90 = False

# Full build cannot run until you explicitly turn this on after review.
APPROVE_FULL_BUILD = False

# Teacher remains off until deterministic dataset is clean.
ENABLE_TEACHER_PILOT = False
TEACHER_MODEL = ""
TEACHER_PILOT_SIZE = 60
TEACHER_FULL_AFTER_PILOT = False

TEACHER_TASKS = [
    "cause",
    "significance",
    "compare",
    "summary",
    "multihop",
    "verification",
]
TEACHER_DEVICE = "auto"
TEACHER_BATCH_SIZE = 1
TEACHER_MAX_NEW_TOKENS = 512
TEACHER_TEMPERATURE = 0.0
TEACHER_FAILURE_POLICY = "fallback"

RESUME = True

SMOKE_REVIEW_MAX_CHARS = 100_000
SMOKE_REVIEW_RESULTS_PER_CALL = 3
SMOKE_REVIEW_SNIPPET_CHARS = 360

print("CONFIG READY")

## Cell 3 — Mount Drive

Mount Drive để đọc deployment artifacts và lưu output.

In [ ]:
from google.colab import drive
drive.mount(DRIVE_MOUNT)

from pathlib import Path

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)

print("DRIVE READY:", DRIVE_ROOT)

## Cell 4 — Fresh clone + pin commit

Clone `main` mới nhất. Output folder gắn với commit SHA để không resume lẫn code cũ.

In [ ]:
import shutil
import subprocess
from pathlib import Path

repo = Path(REPO_DIR)
if repo.exists():
    shutil.rmtree(repo)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True,
)

REPO_HEAD = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
    text=True,
).strip()

status = subprocess.check_output(
    ["git", "-C", REPO_DIR, "status", "--porcelain"],
    text=True,
)

short = REPO_HEAD[:8]
tag = f"_{RUN_TAG.strip()}" if RUN_TAG.strip() else ""

RUN_NAME = f"{BASE_RUN_NAME}_{short}{tag}"
OUTPUT_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
SMOKE_PUBLIC_DIR = f"{DRIVE_ROOT}/smoke_public_v4_3_{short}{tag}"
SMOKE_CUSTOM_DIR = f"{DRIVE_ROOT}/smoke_custom_v4_3_{short}{tag}"
EXTENDED_SMOKE_DIR = f"{DRIVE_ROOT}/smoke_custom90_v4_3_{short}{tag}"

print("HEAD:", REPO_HEAD)
print("Worktree clean:", status.strip() == "")
print("OUTPUT_DIR:", OUTPUT_DIR)

if status.strip():
    raise RuntimeError("Fresh clone is unexpectedly dirty.")

## Cell 5 — Readiness check cho final-quality Codex pass

Cell này xác nhận repo mới đã có các lớp bảo vệ cuối:

- person classifier được harden;
- entity relevance filter;
- compare target isolation;
- entity/facet-aware sentence selection;
- strict semantic audit cho target mismatch/contamination;
- trajectory-level budget;
- expected/unexpected empty audit;
- citation parser + empty-prefix fix vẫn còn.

Nếu Codex chưa push xong hoặc implement thiếu, cell sẽ báo `MISSING`.

In [ ]:
from pathlib import Path

files = {
    "builder": Path(REPO_DIR) / "training/trajectory_dataset/builders/custom_history.py",
    "cli": Path(REPO_DIR) / "training/trajectory_dataset/cli.py",
    "audit": Path(REPO_DIR) / "training/trajectory_dataset/audit.py",
    "citations": Path(REPO_DIR) / "training/trajectory_dataset/citations.py",
    "preprocess": Path(REPO_DIR) / "training/trajectory_dataset/preprocess.py",
    "common": Path(REPO_DIR) / "training/trajectory_dataset/adapters/common.py",
    "split": Path(REPO_DIR) / "training/trajectory_dataset/split.py",
}

for name, path in files.items():
    if not path.exists():
        raise FileNotFoundError(path)

src = {name: path.read_text(encoding="utf-8") for name, path in files.items()}

builder_l = src["builder"].casefold()
audit_l = src["audit"].casefold()

checks = {
    "builder_v4": '"builder_version": "v4"' in src["builder"],
    "subject_classification": "classify_subject" in src["builder"],
    "location_thi_tran": "thị trấn" in src["builder"] or "thi tran" in builder_l,
    "trajectory_budget": (
        "trajectory_observation_char_budget" in src["builder"]
        and "trajectory_observation_char_budget" in src["cli"]
    ),
    "role_aware_empty_audit": (
        "unexpected_empty" in audit_l
        or "expected_empty" in audit_l
    ),
    "canonical_citations": "extract_evidence_citations" in src["citations"],
    "empty_prefix_fix": 'if not messages:' in src["preprocess"] and 'return ""' in src["preprocess"],
    "agent_flan_conversation": 'row.get("conversation")' in src["common"],
    "source_groups": "source_groups" in src["split"],

    # Final-quality requirements
    "entity_relevance_filter": (
        "relevant_to_target" in builder_l
        or "result_relevant" in builder_l
        or "entity_relevance" in builder_l
    ),
    "entity_aware_sentence_selection": (
        "target_title" in src["builder"]
        and ("sentence" in builder_l and "target" in builder_l)
    ),
    "compare_target_isolation": (
        "target_a" in src["builder"]
        and "target_b" in src["builder"]
        and ("relevant" in builder_l or "filter" in builder_l)
    ),
    "strict_semantic_audit": any(
        key in src["audit"]
        for key in (
            "subject_type_mismatch",
            "observation_target_mismatch",
            "final_answer_target_mismatch",
            "compare_target_contamination",
        )
    ),
}

for name, ok in checks.items():
    print(f"{name:34s}", "PASS" if ok else "MISSING")

missing = [name for name, ok in checks.items() if not ok]
if missing:
    raise RuntimeError(
        "Repo chưa khớp final-quality requirements. Missing: " + ", ".join(missing)
    )

print("\nFINAL-QUALITY READINESS: PASS")

## Cell 6 — Minimal dependencies

Không cài training stack đầy đủ; không reinstall torch/numpy.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    check=False,
)

packages = [
    "datasets>=4.0.0",
    "transformers==4.57.6",
    "sentence-transformers==5.6.1",
    "sentencepiece==0.2.2",
    "accelerate>=1.10.0",
    "faiss-cpu==1.15.0",
    "bm25s==0.3.10",
    "pydantic>=2.10,<3",
    "pydantic-settings>=2.10,<3",
    "fastapi>=0.115,<1",
    "python-multipart>=0.0.20",
    "tqdm>=4.67.0",
    "requests>=2.32.0",
]

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        *packages,
    ],
    check=True,
)

print("MINIMAL DEPENDENCIES INSTALLED")

## Cell 7 — Import preflight + auto GPU performance profile

Kiểm package + CUDA, **chưa load retrieval model**.

Cell này cũng resolve `RERANK_BATCH_SIZE` theo VRAM trước Cell 10 ghi build context:

- < 14 GB → 4
- 14–21 GB → 8
- 22–37 GB → 16
- 38–69 GB → 24
- ≥ 70 GB → 32

Mục tiêu là giảm số forward pass của BGE/cross-encoder reranker mà không đổi `TOP_K` hay evidence budget.

**GPU khuyến nghị:** L4/A10 24 GB. T4 vẫn chạy được; A100/H100 chỉ đáng dùng nếu bạn ưu tiên thời gian hơn chi phí/quota.


In [ ]:
import numpy as np
import torch
import faiss
import bm25s
import transformers
import sentence_transformers
import datasets

print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

if USE_GPU_FOR_RETRIEVAL and not torch.cuda.is_available():
    raise RuntimeError("GPU retrieval requested but CUDA unavailable.")

def _recommended_rerank_batch(total_vram_gb: float) -> int:
    # Conservative profiles because the project keeps embedding + reranker
    # components in the same retrieval subprocess.
    if total_vram_gb >= 70:
        return 64
    if total_vram_gb >= 38:
        return 48
    if total_vram_gb >= 22:
        return 32
    if total_vram_gb >= 14:
        return 16
    return 4

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = props.name
    total_vram_gb = props.total_memory / (1024 ** 3)

    if AUTO_TUNE_RERANK_BATCH:
        RERANK_BATCH_SIZE = min(
            _recommended_rerank_batch(total_vram_gb),
            int(RERANK_BATCH_SIZE_CAP),
        )

    print(f"GPU: {gpu_name}")
    print(f"Total VRAM: {total_vram_gb:.1f} GB")
    print(f"AUTO_TUNE_RERANK_BATCH: {AUTO_TUNE_RERANK_BATCH}")
    print(f"Effective RERANK_BATCH_SIZE: {RERANK_BATCH_SIZE}")
else:
    print("CPU retrieval mode")
    print(f"Effective RERANK_BATCH_SIZE: {RERANK_BATCH_SIZE}")

print("IMPORT PREFLIGHT: PASS")


## Cell 8 — Helper functions

Khác V4.2 ở điểm quan trọng: `run_cmd()` có thể chấp nhận exit code 2.

CLI `audit` dùng exit code 2 khi report `valid=false`. Đây là **quality gate**, không phải Python crash. Notebook sẽ đọc report rồi hiển thị `STOP/FIX`, thay vì tạo traceback khó hiểu.

In [ ]:
import os
import sys
import subprocess
import json
from pathlib import Path

ENV = os.environ.copy()
ENV["PYTHONPATH"] = REPO_DIR + os.pathsep + ENV.get("PYTHONPATH", "")
PY = sys.executable

def run_cmd(cmd, title, *, allowed_returncodes=(0,)):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)
    print(" ".join(str(x) for x in cmd))
    print()

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=REPO_DIR,
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()
    print(f"\n{return_code_label(rc)}")

    if rc not in set(allowed_returncodes):
        raise RuntimeError(
            f"{title} FAILED with exit code {rc}. Full child output is above."
        )
    return rc

def return_code_label(rc):
    if rc == 0:
        return "PROCESS EXIT=0 (PASS)"
    if rc == 2:
        return "PROCESS EXIT=2 (AUDIT QUALITY-GATE FAIL; report will be reviewed below)"
    return f"PROCESS EXIT={rc}"

def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

def read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def gpu_status():
    subprocess.run(["nvidia-smi"], check=False)
    raw = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=memory.total,memory.used,memory.free,name",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    ).strip().splitlines()[0]

    total_s, used_s, free_s, name = [x.strip() for x in raw.split(",", 3)]
    status = {
        "name": name,
        "total_mb": int(total_s),
        "used_mb": int(used_s),
        "free_mb": int(free_s),
    }
    print(status)
    return status

def assert_gpu_headroom():
    if not USE_GPU_FOR_RETRIEVAL:
        return
    status = gpu_status()
    if status["free_mb"] < MIN_FREE_GPU_MB:
        raise RuntimeError(
            f"Only {status['free_mb']} MB VRAM free. Restart runtime before RAG load."
        )

def token_danger(report):
    tok = report.get("tokenizer") or {}
    return {
        "initial_user_lost": int(tok.get("rows_initial_user_lost", 0)),
        "tool_call_supervision_lost": int(tok.get("rows_any_tool_call_supervision_lost", 0)),
        "final_assistant_lost": int(tok.get("rows_final_assistant_supervision_lost", 0)),
        "all_assistant_supervision_lost": int(tok.get("rows_all_assistant_supervision_lost", 0)),
    }

print("HELPERS READY")

## Cell 9 — Validate deployment artifacts

Read-only check cho corpus/FAISS/BM25/config.

In [ ]:
corpus_file = Path(CORPUS_PATH) / "vn_history_rag_chunks_enriched.jsonl"

if not corpus_file.exists():
    matches = list(Path("/content/drive/MyDrive").rglob("vn_history_rag_chunks_enriched.jsonl"))
    if len(matches) == 1:
        corpus_file = matches[0]
        CORPUS_PATH = str(corpus_file.parent)
        DEPLOYMENT_ROOT = str(corpus_file.parent.parent)
        print("Auto-detected corpus:", corpus_file)
    elif not matches:
        raise FileNotFoundError("Cannot find enriched corpus.")
    else:
        for i, p in enumerate(matches[:20], 1):
            print(i, p)
        raise RuntimeError("Multiple corpus candidates; set CORPUS_PATH manually.")

deployment = Path(CORPUS_PATH).parent

required = [
    deployment / "corpus" / "vn_history_rag_chunks_enriched.jsonl",
    deployment / "retrieval" / "faiss" / "chunks.index",
    deployment / "retrieval" / "faiss" / "manifest.json",
    deployment / "retrieval" / "bm25s_index",
    deployment / "retrieval" / "bm25s_index" / "phase9_manifest.json",
    deployment / "config" / "inference_config.json",
    deployment / "manifest.json",
]

missing = []
for p in required:
    ok = p.exists()
    print(("OK      " if ok else "MISSING "), p)
    if not ok:
        missing.append(str(p))

if missing:
    raise FileNotFoundError("Missing artifacts:\n" + "\n".join(missing))

print("ARTIFACT CHECK: PASS")

## Cell 10 — Build context

Lưu commit + config để resume không bị lẫn code/config.

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

context_path = Path(OUTPUT_DIR) / "colab_build_context.json"

context = {
    "repo_head": REPO_HEAD,
    "corpus_path": str(Path(CORPUS_PATH)),
    "top_k": TOP_K,
    "rerank_batch_size": RERANK_BATCH_SIZE,
    "auto_tune_rerank_batch": AUTO_TUNE_RERANK_BATCH,
    "rerank_batch_size_cap": RERANK_BATCH_SIZE_CAP,
    "checkpoint_every": CHECKPOINT_EVERY,
    "public_checkpoint_every": PUBLIC_CHECKPOINT_EVERY,
    "max_corpus_records": MAX_CORPUS_RECORDS,
    "observation_char_budget": OBSERVATION_CHAR_BUDGET,
    "trajectory_observation_char_budget": TRAJECTORY_OBSERVATION_CHAR_BUDGET,
    "max_result_text_chars": MAX_RESULT_TEXT_CHARS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "seed": SEED,
    "custom_counts": CUSTOM_COUNTS,
    "mix_ratios": MIX_RATIOS,
}

if context_path.exists():
    old = load_json(context_path)
    if old != context:
        raise RuntimeError(
            "Existing run directory has different config. Change RUN_TAG or remove the run."
        )
else:
    context_path.write_text(
        json.dumps(context, ensure_ascii=False, indent=2, sort_keys=True),
        encoding="utf-8",
    )

print(json.dumps(context, ensure_ascii=False, indent=2))

## Cell 11 — Corpus inspect

Read-only subject/facet preview.

In [ ]:
run_cmd(
    [
        PY, "-m", "training.trajectory_dataset.cli", "inspect",
        "--corpus-path", CORPUS_PATH,
        "--max-samples", "5000",
    ],
    "CORPUS INSPECT",
)

## Cell 12 — Public smoke 20/source

Xác nhận 3 public adapters chạy thật.

In [ ]:
import shutil

public_smoke = Path(SMOKE_PUBLIC_DIR)
if public_smoke.exists():
    shutil.rmtree(public_smoke)
public_smoke.mkdir(parents=True, exist_ok=True)

public_jobs = [
    (
        "AGENT-FLAN SMOKE",
        [
            PY, "-m", "training.trajectory_dataset.cli", "normalize-public",
            "--source", "agent_flan",
            "--split", "agent_instruct_react",
            "--max-samples", "20",
            "--max-attempts", "400",
            "--cache-dir", HF_CACHE_DIR,
            "--output", str(public_smoke / "agent_flan.jsonl"),
            "--no-include-reasoning",
        ],
        public_smoke / "agent_flan.jsonl",
    ),
    (
        "MULTIHOP PUBLIC SMOKE",
        [
            PY, "-m", "training.trajectory_dataset.cli", "normalize-public",
            "--source", "multihop",
            "--split", "train",
            "--max-samples", "20",
            "--max-attempts", "400",
            "--cache-dir", HF_CACHE_DIR,
            "--output", str(public_smoke / "multihop.jsonl"),
            "--no-include-reasoning",
        ],
        public_smoke / "multihop.jsonl",
    ),
    (
        "VIETNAM HISTORY PUBLIC SMOKE",
        [
            PY, "-m", "training.trajectory_dataset.cli", "normalize-public",
            "--source", "vietnam_history",
            "--split", "train",
            "--history-mode", HISTORY_MODE,
            "--max-samples", "20",
            "--max-attempts", "400",
            "--cache-dir", HF_CACHE_DIR,
            "--output", str(public_smoke / "vietnam_history.jsonl"),
            "--no-include-reasoning",
        ],
        public_smoke / "vietnam_history.jsonl",
    ),
]

for title, cmd, output in public_jobs:
    run_cmd(cmd, title)
    if count_jsonl(output) != 20:
        raise RuntimeError(f"{title}: expected 20 rows.")

print("PUBLIC SMOKE: PASS")

## Cell 13 — Public token audit

Audit tokenizer-only. Exit code 2 được xử lý như quality gate thay vì traceback.

In [ ]:
PUBLIC_TOKEN_REPORTS = {}
PUBLIC_TOKEN_PASS = True

for name in ("agent_flan", "multihop", "vietnam_history"):
    src = public_smoke / f"{name}.jsonl"
    report_path = public_smoke / f"{name}.token_audit.json"

    run_cmd(
        [
            PY, "-m", "training.trajectory_dataset.cli", "audit",
            "--input", str(src),
            "--tokenizer-model-id", "Qwen/Qwen3-8B",
            "--max-seq-length", str(MAX_SEQ_LENGTH),
            "--output", str(report_path),
        ],
        f"{name.upper()} TOKEN AUDIT",
        allowed_returncodes=(0, 2),
    )

    report = load_json(report_path)
    danger = token_danger(report)
    print(name, "token danger:", danger)
    PUBLIC_TOKEN_REPORTS[name] = report_path

    if any(danger.values()):
        PUBLIC_TOKEN_PASS = False

print("PUBLIC_TOKEN_PASS =", PUBLIC_TOKEN_PASS)

## Cell 14 — GPU headroom

Kiểm VRAM trước project RAG.

In [ ]:
assert_gpu_headroom()

## Cell 15 — Custom smoke 30

Smoke 30 với project retrieval thật và **trajectory-level budget**.

Nếu Codex dùng đúng CLI requirement mới, flag này sẽ có:
`--trajectory-observation-char-budget 5200`.

In [ ]:
smoke = Path(SMOKE_CUSTOM_DIR)
if smoke.exists():
    shutil.rmtree(smoke)
smoke.mkdir(parents=True, exist_ok=True)

SMOKE_COUNTS = {
    "factual": 4,
    "cause": 4,
    "significance": 3,
    "compare": 4,
    "summary": 3,
    "multihop": 4,
    "verification": 3,
    "hard_negative": 3,
    "insufficient_evidence": 2,
}
assert sum(SMOKE_COUNTS.values()) == 30

device = "cuda" if (USE_GPU_FOR_RETRIEVAL and torch.cuda.is_available()) else "cpu"

cmd = [
    PY, "-m", "training.trajectory_dataset.cli", "build-custom",
    "--corpus-path", CORPUS_PATH,
    "--output-dir", SMOKE_CUSTOM_DIR,
    "--retrieval-backend", "project",
    "--device", device,
    "--top-k", str(TOP_K),
    "--rerank-batch-size", str(RERANK_BATCH_SIZE),
    "--observation-char-budget", str(OBSERVATION_CHAR_BUDGET),
    "--trajectory-observation-char-budget", str(TRAJECTORY_OBSERVATION_CHAR_BUDGET),
    "--max-result-text-chars", str(MAX_RESULT_TEXT_CHARS),
    "--max-corpus-records", "5000",
    "--teacher-backend", "none",
    "--checkpoint-every", "5",
    "--seed", str(SEED),
    "--no-include-no-tool",
]

for task, count in SMOKE_COUNTS.items():
    cmd += ["--num-" + task.replace("_", "-"), str(count)]

run_cmd(cmd, "CUSTOM V4 POST-CODEX SMOKE 30")

smoke_file = Path(SMOKE_CUSTOM_DIR) / "custom_history.jsonl"
smoke_count = count_jsonl(smoke_file)

print("Smoke rows:", smoke_count)
if smoke_count != 30:
    raise RuntimeError(f"Expected 30 rows, got {smoke_count}")

print("GPU after retrieval subprocess exits:")
gpu_status()

## Cell 16 — Smoke strict semantic + token + relevance audit

Audit mới phải bắt không chỉ lỗi schema/token/empty, mà cả **entity quality**.

Nếu CLI trả exit code `2`, notebook vẫn không crash; nó đọc report để xem:
- subject type mismatch;
- target/evidence mismatch;
- compare contamination;
- unexpected required empty;
- token safety.

Cell sau mới quyết định `SMOKE_GATE_PASS`.

In [ ]:
custom_smoke_audit = Path(SMOKE_CUSTOM_DIR) / "custom_smoke_audit.json"

audit_rc = run_cmd(
    [
        PY, "-m", "training.trajectory_dataset.cli", "audit",
        "--input", str(smoke_file),
        "--strict-custom",
        "--tokenizer-model-id", "Qwen/Qwen3-8B",
        "--max-seq-length", str(MAX_SEQ_LENGTH),
        "--output", str(custom_smoke_audit),
    ],
    "CUSTOM SMOKE STRICT + TOKEN AUDIT",
    allowed_returncodes=(0, 2),
)

smoke_audit = load_json(custom_smoke_audit)

print("\nAudit valid:", smoke_audit.get("valid"))
print("Issues:", smoke_audit.get("issues"))
print("Observation chars:", smoke_audit.get("observation_chars"))
print("Raw empty rate:", smoke_audit.get("empty_tool_result_rate"))
print("Token stats:", (smoke_audit.get("tokenizer") or {}).get("rendered_tokens"))
print("Token danger:", token_danger(smoke_audit))

## Cell 17 — Clean reliability gate (không traceback)

Cell này quyết định `SMOKE_GATE_PASS`.

Nó **không raise error**. Nếu còn vấn đề, nó in rõ `STOP BEFORE FULL BUILD`.

Gate dựa trên:
- CLI semantic validity;
- unexpected required empty results;
- dangerous token truncation;
- public token audit.

Expected-empty như `insufficient_evidence` hay hard-negative wrong-facet không tự làm fail.

In [ ]:
issues = smoke_audit.get("issues") or {}
danger = token_danger(smoke_audit)

def get_int(report, *keys):
    for key in keys:
        value = report.get(key)
        if isinstance(value, int):
            return value
        if isinstance(value, float):
            return int(value)
    return None

unexpected_empty = get_int(
    smoke_audit,
    "unexpected_empty_tool_results",
    "unexpected_empty_results",
)
expected_empty = get_int(
    smoke_audit,
    "expected_empty_tool_results",
    "expected_empty_results",
)

if unexpected_empty is None:
    unexpected_empty = int(
        issues.get("unexpected_empty_tool_results", 0)
        or issues.get("unexpected_required_empty_results", 0)
        or 0
    )

SEMANTIC_ISSUE_KEYS = (
    "subject_type_mismatch",
    "observation_target_mismatch",
    "final_answer_target_mismatch",
    "compare_target_contamination",
)

semantic_issue_counts = {
    key: int(issues.get(key, 0) or 0)
    for key in SEMANTIC_ISSUE_KEYS
}

SMOKE_GATE_REASONS = []

if not bool(smoke_audit.get("valid", False)):
    SMOKE_GATE_REASONS.append("audit valid=false")

if unexpected_empty > 0:
    SMOKE_GATE_REASONS.append(
        f"{unexpected_empty} unexpected required empty retrieval result(s)"
    )

for key, count in semantic_issue_counts.items():
    if count > 0:
        SMOKE_GATE_REASONS.append(f"{key}={count}")

if any(danger.values()):
    SMOKE_GATE_REASONS.append(f"dangerous 4096-token truncation: {danger}")

if not PUBLIC_TOKEN_PASS:
    SMOKE_GATE_REASONS.append("public token smoke is unsafe")

SMOKE_GATE_PASS = len(SMOKE_GATE_REASONS) == 0

print("=" * 100)
print("SMOKE RELIABILITY GATE")
print("=" * 100)
print("Expected empty results:", expected_empty)
print("Unexpected required empty results:", unexpected_empty)
print("Semantic issue counts:", semantic_issue_counts)
print("Token danger:", danger)
print("Audit issues:", issues)

if SMOKE_GATE_PASS:
    print("\nSMOKE_GATE_PASS=True")
    print("Automatic gate passed. Still run targeted sanity + send review bundle.")
else:
    print("\nSMOKE_GATE_PASS=False")
    print("STOP BEFORE FULL BUILD.")
    for reason in SMOKE_GATE_REASONS:
        print(" -", reason)

print("\nNo traceback is intentionally raised in this gate cell.")

## Cell 18 — Human-readable 30 samples

In toàn bộ 30 samples ngắn gọn để tự nhìn.

In [ ]:
rows = read_jsonl(smoke_file)

for i, row in enumerate(rows, 1):
    prov = row.get("provenance") or {}
    print("\n" + "=" * 105)
    print(
        f"{i:02d}/30 task={row.get('task_type')} "
        f"type={prov.get('subject_type')} "
        f"title={prov.get('primary_title')}"
    )

    if prov.get("secondary_title"):
        print(
            "secondary:",
            prov.get("secondary_title"),
            "| type=",
            prov.get("secondary_subject_type"),
        )

    claim = prov.get("concrete_claim") or prov.get("synthetic_claim")
    if claim:
        print("claim:", claim)

    for msg in row.get("messages", []):
        if msg.get("role") == "user":
            print("\nUSER:", msg.get("content"))

        elif msg.get("role") == "assistant" and msg.get("tool_calls"):
            for call in msg.get("tool_calls") or []:
                print("CALL:", json.dumps(call.get("function"), ensure_ascii=False))

        elif msg.get("role") == "tool":
            try:
                payload = json.loads(msg.get("content") or "[]")
            except Exception:
                payload = []
            print("RESULTS:", len(payload) if isinstance(payload, list) else "?")
            if isinstance(payload, list):
                for item in payload[:3]:
                    if isinstance(item, dict):
                        print(" -", item.get("chunk_id"), "|", item.get("title"))

        elif msg.get("role") == "assistant" and msg.get("content"):
            print("\nFINAL:\n", msg.get("content"))

print("\nHUMAN SMOKE DUMP COMPLETE")

## Cell 18B — Targeted semantic sanity check cho các lỗi smoke cũ

Cell này kiểm nhanh các lỗi từng xuất hiện:

- `Chữ Quốc ngữ` không được là `person`;
- không được có person-question kiểu “là ai / cuộc đời” cho `Chữ Quốc ngữ`;
- evidence cho `Nguyễn Bình` không được drift sang `Nguyễn Bỉnh Khiêm`;
- compare target A/B không được dùng evidence của entity gần tên nhưng khác đối tượng.

Cell **không thay strict audit**, chỉ là sanity độc lập để phát hiện regression dễ nhìn.

In [ ]:
import unicodedata
import re

def norm_text(value):
    value = unicodedata.normalize("NFKD", str(value or "").casefold())
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return " ".join(re.sub(r"[^a-z0-9]+", " ", value).split())

TARGETED_SANITY_ERRORS = []

for row in rows:
    prov = row.get("provenance") or {}
    title = str(prov.get("primary_title") or "")
    subject_type = str(prov.get("subject_type") or "")
    question = next(
        (m.get("content") for m in row.get("messages", []) if m.get("role") == "user"),
        "",
    )

    if norm_text(title) == norm_text("Chữ Quốc ngữ"):
        if subject_type == "person":
            TARGETED_SANITY_ERRORS.append("Chữ Quốc ngữ classified as person")
        qn = norm_text(question)
        if "la ai" in qn or "cuoc doi" in qn or "nhan vat" in qn:
            TARGETED_SANITY_ERRORS.append(
                f"Chữ Quốc ngữ received person-style question: {question}"
            )

    # Inspect tool observations for obvious similar-name contamination.
    for msg in row.get("messages", []):
        if msg.get("role") != "tool":
            continue
        try:
            payload = json.loads(msg.get("content") or "[]")
        except Exception:
            payload = []
        if not isinstance(payload, list):
            continue

        for item in payload:
            if not isinstance(item, dict):
                continue
            item_title = str(item.get("title") or "")
            item_text = str(item.get("text") or "")
            role = str(item.get("retrieval_role") or "")

            if norm_text(title) == norm_text("Nguyễn Bình"):
                blob = norm_text(item_title + " " + item_text)
                if (
                    "nguyen binh" not in blob
                    and "nguyen binh" in norm_text(title)
                    and "nguyen binh khiem" in blob
                ):
                    TARGETED_SANITY_ERRORS.append(
                        f"Nguyễn Bình contaminated by Nguyễn Bỉnh Khiêm in role={role}"
                    )

if TARGETED_SANITY_ERRORS:
    print("TARGETED_SANITY_PASS=False")
    for err in TARGETED_SANITY_ERRORS:
        print(" -", err)
else:
    print("TARGETED_SANITY_PASS=True")

TARGETED_SANITY_PASS = not TARGETED_SANITY_ERRORS

## Cell 19 — `SMOKE_REVIEW_BUNDLE` để gửi ChatGPT

Cell này luôn chạy được, kể cả automatic gate hoặc targeted sanity fail.

Copy nguyên block từ BEGIN đến END gửi ChatGPT. Bundle giờ có thêm:
- semantic issue counts;
- targeted sanity result;
- commit/config;
- public token audits;
- strict custom audit;
- 30 samples với tool queries, top results và final answers.

Nếu output bị truncate, file đầy đủ được lưu trên Drive.

In [ ]:
def clip_text(value, limit):
    value = " ".join(str(value or "").split())
    return value if len(value) <= limit else value[:limit] + "...<cut>"

def compact_sample(row):
    prov = row.get("provenance") or {}
    sample = {
        "id": row.get("id"),
        "task": row.get("task_type"),
        "subject_type": prov.get("subject_type"),
        "primary_title": prov.get("primary_title"),
        "secondary_title": prov.get("secondary_title"),
        "secondary_subject_type": prov.get("secondary_subject_type"),
        "claim": prov.get("concrete_claim") or prov.get("synthetic_claim"),
        "question": None,
        "calls": [],
        "final": row.get("messages", [])[-1].get("content") if row.get("messages") else None,
    }

    pending_call = None

    for msg in row.get("messages", []):
        if msg.get("role") == "user" and sample["question"] is None:
            sample["question"] = msg.get("content")

        elif msg.get("role") == "assistant" and msg.get("tool_calls"):
            for call in msg.get("tool_calls") or []:
                fn = call.get("function") or {}
                pending_call = {
                    "name": fn.get("name"),
                    "arguments": fn.get("arguments"),
                    "results": [],
                }
                sample["calls"].append(pending_call)

        elif msg.get("role") == "tool":
            try:
                payload = json.loads(msg.get("content") or "[]")
            except Exception:
                payload = []

            if pending_call is not None and isinstance(payload, list):
                for item in payload[:SMOKE_REVIEW_RESULTS_PER_CALL]:
                    if isinstance(item, dict):
                        pending_call["results"].append({
                            "chunk_id": item.get("chunk_id"),
                            "title": item.get("title"),
                            "retrieval_role": item.get("retrieval_role"),
                            "text": clip_text(
                                item.get("text"),
                                SMOKE_REVIEW_SNIPPET_CHARS,
                            ),
                        })

    return sample

public_summary = {
    name: load_json(path).get("tokenizer")
    for name, path in PUBLIC_TOKEN_REPORTS.items()
}

bundle = {
    "repo_head": REPO_HEAD,
    "config": context,
    "automatic_gate": {
        "pass": SMOKE_GATE_PASS,
        "reasons": SMOKE_GATE_REASONS,
        "expected_empty": expected_empty,
        "unexpected_empty": unexpected_empty,
        "token_danger": danger,
        "semantic_issue_counts": semantic_issue_counts,
        "targeted_sanity_pass": TARGETED_SANITY_PASS,
        "targeted_sanity_errors": TARGETED_SANITY_ERRORS,
    },
    "public_token_audits": public_summary,
    "custom_smoke_audit": smoke_audit,
    "samples": [compact_sample(row) for row in rows],
}

bundle_json = json.dumps(bundle, ensure_ascii=False, indent=2)

bundle_path = Path(SMOKE_CUSTOM_DIR) / "SMOKE_REVIEW_BUNDLE.txt"
bundle_path.write_text(
    "===== SMOKE_REVIEW_BUNDLE BEGIN =====\n"
    + bundle_json
    + "\n===== SMOKE_REVIEW_BUNDLE END =====\n",
    encoding="utf-8",
)

print("===== SMOKE_REVIEW_BUNDLE BEGIN =====")
print(bundle_json[:SMOKE_REVIEW_MAX_CHARS])
if len(bundle_json) > SMOKE_REVIEW_MAX_CHARS:
    print(
        "\n... notebook display truncated ...\n"
        f"Full bundle: {bundle_path}"
    )
print("===== SMOKE_REVIEW_BUNDLE END =====")
print("\nFull bundle saved:", bundle_path)

## Cell 20 — Optional smoke 90

Chỉ bật nếu smoke 30 sạch nhưng muốn thêm confidence.

Nếu `SMOKE_GATE_PASS=False`, cell sẽ tự skip.

In [ ]:
if RUN_EXTENDED_SMOKE_90 and SMOKE_GATE_PASS:
    ext = Path(EXTENDED_SMOKE_DIR)
    if ext.exists():
        shutil.rmtree(ext)
    ext.mkdir(parents=True, exist_ok=True)

    assert_gpu_headroom()

    EXT_COUNTS = {
        "factual": 10,
        "cause": 10,
        "significance": 10,
        "compare": 10,
        "summary": 10,
        "multihop": 10,
        "verification": 10,
        "hard_negative": 10,
        "insufficient_evidence": 10,
    }

    cmd = [
        PY, "-m", "training.trajectory_dataset.cli", "build-custom",
        "--corpus-path", CORPUS_PATH,
        "--output-dir", EXTENDED_SMOKE_DIR,
        "--retrieval-backend", "project",
        "--device", device,
        "--top-k", str(TOP_K),
        "--rerank-batch-size", str(RERANK_BATCH_SIZE),
        "--observation-char-budget", str(OBSERVATION_CHAR_BUDGET),
        "--trajectory-observation-char-budget", str(TRAJECTORY_OBSERVATION_CHAR_BUDGET),
        "--max-result-text-chars", str(MAX_RESULT_TEXT_CHARS),
        "--max-corpus-records", "8000",
        "--teacher-backend", "none",
        "--checkpoint-every", "10",
        "--seed", str(SEED + 1000),
        "--no-include-no-tool",
    ]

    for task, count in EXT_COUNTS.items():
        cmd += ["--num-" + task.replace("_", "-"), str(count)]

    run_cmd(cmd, "EXTENDED SMOKE 90")

    ext_file = ext / "custom_history.jsonl"
    ext_audit = ext / "extended_smoke_audit.json"

    run_cmd(
        [
            PY, "-m", "training.trajectory_dataset.cli", "audit",
            "--input", str(ext_file),
            "--strict-custom",
            "--tokenizer-model-id", "Qwen/Qwen3-8B",
            "--max-seq-length", str(MAX_SEQ_LENGTH),
            "--output", str(ext_audit),
        ],
        "EXTENDED SMOKE AUDIT",
        allowed_returncodes=(0, 2),
    )

    print(json.dumps(load_json(ext_audit), ensure_ascii=False, indent=2))
else:
    print(
        "Extended smoke skipped. "
        f"RUN_EXTENDED_SMOKE_90={RUN_EXTENDED_SMOKE_90}, "
        f"SMOKE_GATE_PASS={SMOKE_GATE_PASS}"
    )


# ⛔ Manual review checkpoint

Sau Cell 19:

1. gửi `SMOKE_REVIEW_BUNDLE` cho ChatGPT;
2. đợi review;
3. nếu được báo **GO FULL**, quay lại Cell 2 và đổi:

```python
APPROVE_FULL_BUILD = True
```

rồi chạy lại notebook từ đầu hoặc ít nhất chạy lại các cell phụ thuộc config/context theo đúng run mới.

Phần full dưới đây chỉ chạy khi:

```python
SMOKE_GATE_PASS is True
and APPROVE_FULL_BUILD is True
```

nếu không nó chỉ in `SKIPPED`.


## Cell 21 — Full-build permission

Full build chỉ được unlock nếu:

```python
SMOKE_GATE_PASS is True
TARGETED_SANITY_PASS is True
APPROVE_FULL_BUILD is True
```

Tức là automatic audit sạch + sanity sạch + bạn đã được review thủ công và chủ động bật full.

In [ ]:
APPROVE_FULL_BUILD = True
FULL_BUILD_ALLOWED = bool(
    SMOKE_GATE_PASS
    and TARGETED_SANITY_PASS
    and APPROVE_FULL_BUILD
)

print("SMOKE_GATE_PASS =", SMOKE_GATE_PASS)
print("TARGETED_SANITY_PASS =", TARGETED_SANITY_PASS)
print("APPROVE_FULL_BUILD =", APPROVE_FULL_BUILD)
print("FULL_BUILD_ALLOWED =", FULL_BUILD_ALLOWED)

if not FULL_BUILD_ALLOWED:
    print("FULL BUILD IS LOCKED — cells below will skip heavy work.")

## Cell 22 — Full public normalization

Chỉ chạy khi full build được unlock.

### Agent-FLAN
- dùng `--split auto` cho full pool;
- truyền `--final-max-samples FINAL_MAX_SAMPLES` để CLI trả `final_mix_gate`;
- nếu checkpoint cũ là single-split, chỉ regenerate ba file `intermediate/agent_flan*`;
- ưu tiên 700 rows; chỉ cho degraded pass khi chính `final_mix_gate` của CLI xác nhận pool an toàn.

Multi-hop và Vietnam History giữ nguyên explicit split và exact-count gate.


In [ ]:
if FULL_BUILD_ALLOWED:
    import json
    import subprocess

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    from training.trajectory_dataset.cli import AGENT_FLAN_COMPATIBLE_SPLITS
    from training.trajectory_dataset.notebook_integration import (
        evaluate_agent_flan_notebook_gate,
        prepare_agent_flan_pooled_resume,
    )

    intermediate = Path(OUTPUT_DIR) / "intermediate"
    intermediate.mkdir(parents=True, exist_ok=True)

    agent_flan_out = intermediate / "agent_flan.jsonl"
    multihop_out = intermediate / "multihop.jsonl"
    vietnam_history_out = intermediate / "vietnam_history.jsonl"

    compatible_splits = list(AGENT_FLAN_COMPATIBLE_SPLITS)
    final_mix_minimum = int(FINAL_MAX_SAMPLES * MIX_RATIOS["agent_flan"])
    print("Agent-FLAN source mode: pooled")
    print("Agent-FLAN splits:")
    for split_name in compatible_splits:
        print("  -", split_name)
    print("Pool target:", AGENT_FLAN_MAX)
    print("Final mix minimum:", final_mix_minimum)

    if RESUME:
        cleanup = prepare_agent_flan_pooled_resume(
            intermediate,
            expected_splits=compatible_splits,
        )
        if cleanup["regenerated"]:
            print("STALE AGENT-FLAN STATE:", cleanup["reason"])
            print("Regenerating ONLY Agent-FLAN intermediate/state files:")
            for removed_path in cleanup["removed"]:
                print("  removed:", removed_path)

    agent_cmd = [
        PY, "-m", "training.trajectory_dataset.cli", "normalize-public",
        "--source", "agent_flan",
        "--split", "auto",
        "--max-samples", str(AGENT_FLAN_MAX),
        "--max-attempts", str(max(AGENT_FLAN_MAX * 15, 3000)),
        "--cache-dir", HF_CACHE_DIR,
        "--output", str(agent_flan_out),
        "--no-include-reasoning",
        "--checkpoint-every", str(PUBLIC_CHECKPOINT_EVERY),
        "--final-max-samples", str(FINAL_MAX_SAMPLES),
    ]
    if RESUME:
        agent_cmd.append("--resume")

    print("\n" + "=" * 110)
    print("FULL AGENT-FLAN (AUTO POOL)")
    print("=" * 110)
    print(" ".join(str(value) for value in agent_cmd))
    agent_proc = subprocess.run(
        [str(value) for value in agent_cmd],
        cwd=REPO_DIR,
        env=ENV,
        stdout=subprocess.PIPE,
        text=True,
        check=False,
    )
    print(agent_proc.stdout)
    if agent_proc.returncode not in {0, 2}:
        raise RuntimeError(
            "FULL AGENT-FLAN FAILED with exit code "
            f"{agent_proc.returncode}. Child stderr is above."
        )
    try:
        agent_report = json.loads(agent_proc.stdout)
    except json.JSONDecodeError as exc:
        raise RuntimeError("FULL AGENT-FLAN did not return its JSON pool report.") from exc

    print("AGENT-FLAN POOL RESULT")
    for key in (
        "written",
        "target_reached",
        "source_exhausted",
        "source_splits",
        "rejected",
        "rejected_by_reason",
        "final_mix_gate",
    ):
        print(f"{key} =", json.dumps(agent_report.get(key), ensure_ascii=False, sort_keys=True))

    agent_status = evaluate_agent_flan_notebook_gate(
        agent_report,
        pool_target=AGENT_FLAN_MAX,
    )
    if agent_status == "PASS":
        print(f"FULL AGENT-FLAN: PASS ({agent_report['written']}/{AGENT_FLAN_MAX})")
    else:
        print(
            "FULL AGENT-FLAN: DEGRADED POOL PASS "
            f"({agent_report['written']} rows; final_mix_gate valid)"
        )

    jobs = [
        (
            "FULL MULTIHOP PUBLIC",
            [
                PY, "-m", "training.trajectory_dataset.cli", "normalize-public",
                "--source", "multihop",
                "--split", "train",
                "--max-samples", str(MULTIHOP_MAX),
                "--max-attempts", str(max(MULTIHOP_MAX * 15, 3000)),
                "--cache-dir", HF_CACHE_DIR,
                "--output", str(multihop_out),
                "--no-include-reasoning",
                "--checkpoint-every", str(PUBLIC_CHECKPOINT_EVERY),
            ],
            multihop_out,
            MULTIHOP_MAX,
        ),
        (
            "FULL VIETNAM HISTORY PUBLIC",
            [
                PY, "-m", "training.trajectory_dataset.cli", "normalize-public",
                "--source", "vietnam_history",
                "--split", "train",
                "--history-mode", HISTORY_MODE,
                "--max-samples", str(VIETNAM_HISTORY_MAX),
                "--max-attempts", str(max(VIETNAM_HISTORY_MAX * 15, 3000)),
                "--cache-dir", HF_CACHE_DIR,
                "--output", str(vietnam_history_out),
                "--no-include-reasoning",
                "--checkpoint-every", str(PUBLIC_CHECKPOINT_EVERY),
            ],
            vietnam_history_out,
            VIETNAM_HISTORY_MAX,
        ),
    ]

    for title, cmd, output, expected in jobs:
        if RESUME:
            cmd.append("--resume")
        run_cmd(cmd, title)
        actual = count_jsonl(output)
        if actual != expected:
            raise RuntimeError(f"{title}: expected {expected} rows, got {actual}.")

    print("FULL PUBLIC NORMALIZATION: PASS")
else:
    print("SKIPPED: full public normalization")


## Cell 23 — Full custom 2.200

Đây thường là cell tốn thời gian nhất vì dùng project retrieval thật cho nhiều query.

Performance version giữ nguyên quality settings nhưng:

- dùng `RERANK_BATCH_SIZE` đã auto-tune ở Cell 7;
- checkpoint mỗi `CHECKPOINT_EVERY=100` rows thay vì 25 để giảm Drive I/O;
- **không** chạy nhiều `build-custom` subprocess song song trên cùng GPU/output folder, vì dễ tranh VRAM và làm hỏng resume/checkpoint.

Trước khi chạy, cell sẽ in GPU + batch + checkpoint thực tế. Nếu gặp CUDA OOM, giảm rerank batch; không cần giảm `TOP_K` hay observation budget.


In [ ]:
if FULL_BUILD_ALLOWED:
    assert_gpu_headroom()

    print("FULL CUSTOM PERFORMANCE CONFIG")
    print("device =", device)
    print("RERANK_BATCH_SIZE =", RERANK_BATCH_SIZE)
    print("CHECKPOINT_EVERY =", CHECKPOINT_EVERY)
    print("TOP_K =", TOP_K)

    cmd = [
        PY, "-m", "training.trajectory_dataset.cli", "build-custom",
        "--corpus-path", CORPUS_PATH,
        "--output-dir", OUTPUT_DIR,
        "--retrieval-backend", "project",
        "--device", device,
        "--top-k", str(TOP_K),
        "--rerank-batch-size", str(RERANK_BATCH_SIZE),
        "--observation-char-budget", str(OBSERVATION_CHAR_BUDGET),
        "--trajectory-observation-char-budget", str(TRAJECTORY_OBSERVATION_CHAR_BUDGET),
        "--max-result-text-chars", str(MAX_RESULT_TEXT_CHARS),
        "--max-corpus-records", str(MAX_CORPUS_RECORDS),
        "--teacher-backend", "none",
        "--checkpoint-every", str(CHECKPOINT_EVERY),
        "--seed", str(SEED),
        "--no-include-no-tool",
    ]

    for task, count in CUSTOM_COUNTS.items():
        cmd += ["--num-" + task.replace("_", "-"), str(count)]

    if RESUME:
        cmd.append("--resume")

    run_cmd(cmd, "FULL CUSTOM V4 POST-CODEX")

    custom_base = Path(OUTPUT_DIR) / "custom_history.jsonl"
    if count_jsonl(custom_base) != 2200:
        raise RuntimeError(
            f"Expected 2200 custom rows, got {count_jsonl(custom_base)}"
        )
else:
    print("SKIPPED: full custom")

## Cell 24 — Full custom audit

Không traceback nếu audit exit=2; thay vào đó set `FULL_CUSTOM_GATE_PASS`.

In [ ]:
FULL_CUSTOM_GATE_PASS = False

if FULL_BUILD_ALLOWED:
    full_custom_audit = Path(OUTPUT_DIR) / "custom_history.audit.json"

    run_cmd(
        [
            PY, "-m", "training.trajectory_dataset.cli", "audit",
            "--input", str(custom_base),
            "--strict-custom",
            "--tokenizer-model-id", "Qwen/Qwen3-8B",
            "--max-seq-length", str(MAX_SEQ_LENGTH),
            "--output", str(full_custom_audit),
        ],
        "FULL CUSTOM STRICT + TOKEN AUDIT",
        allowed_returncodes=(0, 2),
    )

    report = load_json(full_custom_audit)
    full_danger = token_danger(report)
    full_issues = report.get("issues") or {}

    full_unexpected = report.get("unexpected_empty_tool_results")
    if not isinstance(full_unexpected, int):
        full_unexpected = int(
            full_issues.get("unexpected_empty_tool_results", 0)
            or full_issues.get("unexpected_required_empty_results", 0)
            or 0
        )

    FULL_CUSTOM_GATE_PASS = (
        bool(report.get("valid", False))
        and not any(full_danger.values())
        and full_unexpected == 0
    )

    print("FULL_CUSTOM_GATE_PASS =", FULL_CUSTOM_GATE_PASS)
    print("Unexpected required empty:", full_unexpected)
    print("Token danger:", full_danger)
    print("Issues:", full_issues)
else:
    print("SKIPPED: full custom audit")

## Cell 25 — Optional teacher pilot

Teacher chỉ được phép chạy nếu full custom deterministic gate PASS.

In [ ]:
teacher_pilot_output = None

if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS and ENABLE_TEACHER_PILOT:
    if not TEACHER_MODEL.strip():
        raise ValueError("Set TEACHER_MODEL.")

    assert_gpu_headroom()

    import random

    eligible = [
        row for row in read_jsonl(custom_base)
        if row.get("task_type") in set(TEACHER_TASKS)
    ]
    eligible = sorted(eligible, key=lambda row: str(row.get("id") or ""))
    rng = random.Random(SEED)
    rng.shuffle(eligible)

    pilot_rows = eligible[:min(TEACHER_PILOT_SIZE, len(eligible))]
    pilot_input = Path(OUTPUT_DIR) / "teacher_pilot.input.jsonl"
    teacher_pilot_output = Path(OUTPUT_DIR) / "teacher_pilot.output.jsonl"

    with pilot_input.open("w", encoding="utf-8") as f:
        for row in pilot_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    cmd = [
        PY, "-m", "training.trajectory_dataset.cli", "enhance-teacher",
        "--input", str(pilot_input),
        "--output", str(teacher_pilot_output),
        "--teacher-backend", "local_hf",
        "--teacher-model", TEACHER_MODEL,
        "--teacher-device", TEACHER_DEVICE,
        "--teacher-batch-size", str(TEACHER_BATCH_SIZE),
        "--max-new-tokens", str(TEACHER_MAX_NEW_TOKENS),
        "--temperature", str(TEACHER_TEMPERATURE),
        "--failure-policy", TEACHER_FAILURE_POLICY,
        "--seed", str(SEED),
    ]
    for task in TEACHER_TASKS:
        cmd += ["--task-type", task]

    run_cmd(cmd, "TEACHER PILOT")
else:
    print("SKIPPED: teacher pilot")

## Cell 26 — Optional full teacher

Mặc định bỏ qua. Chỉ chạy khi pilot đã được review và `TEACHER_FULL_AFTER_PILOT=True`.

In [ ]:
CUSTOM_FOR_MIX = None

if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS:
    CUSTOM_FOR_MIX = custom_base

    if ENABLE_TEACHER_PILOT and TEACHER_FULL_AFTER_PILOT:
        teacher_full = Path(OUTPUT_DIR) / "custom_history.teacher.jsonl"

        cmd = [
            PY, "-m", "training.trajectory_dataset.cli", "enhance-teacher",
            "--input", str(custom_base),
            "--output", str(teacher_full),
            "--teacher-backend", "local_hf",
            "--teacher-model", TEACHER_MODEL,
            "--teacher-device", TEACHER_DEVICE,
            "--teacher-batch-size", str(TEACHER_BATCH_SIZE),
            "--max-new-tokens", str(TEACHER_MAX_NEW_TOKENS),
            "--temperature", str(TEACHER_TEMPERATURE),
            "--failure-policy", TEACHER_FAILURE_POLICY,
            "--seed", str(SEED),
        ]
        for task in TEACHER_TASKS:
            cmd += ["--task-type", task]

        run_cmd(cmd, "FULL TEACHER ENHANCEMENT")
        CUSTOM_FOR_MIX = teacher_full

print("CUSTOM_FOR_MIX:", CUSTOM_FOR_MIX)

## Cell 27 — Mix + dedup

Chỉ mix nếu full custom gate PASS.

In [ ]:
if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS:
    source_files = {
        "custom_history": Path(CUSTOM_FOR_MIX),
        "multi_hop_function_calling": multihop_out,
        "agent_flan": agent_flan_out,
        "vietnam_history_200k": vietnam_history_out,
    }

    counts = {name: count_jsonl(path) for name, path in source_files.items()}
    print("SOURCE COUNTS:", json.dumps(counts, indent=2))

    mixed_path = Path(OUTPUT_DIR) / "mixed.jsonl"
    mix_rejected = Path(OUTPUT_DIR) / "mix_rejected.jsonl"

    cmd = [PY, "-m", "training.trajectory_dataset.cli", "mix"]
    for source_name, path in source_files.items():
        cmd += ["--input", f"{source_name}={path}"]
    for source_name, ratio in MIX_RATIOS.items():
        cmd += ["--ratio", f"{source_name}={ratio}"]

    cmd += [
        "--output", str(mixed_path),
        "--rejected-output", str(mix_rejected),
        "--max-samples", str(FINAL_MAX_SAMPLES),
        "--seed", str(SEED),
    ]

    run_cmd(cmd, "MIX + DEDUP")

    mixed_count = count_jsonl(mixed_path)
    if mixed_count < MIN_ACCEPTABLE_MIXED_ROWS:
        raise RuntimeError(
            f"Mixed rows {mixed_count} < minimum {MIN_ACCEPTABLE_MIXED_ROWS}"
        )
else:
    print("SKIPPED: mix")

## Cell 28 — Canonical validation

Final schema validation.

In [ ]:
if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS:
    validated_path = Path(OUTPUT_DIR) / "mixed.validated.jsonl"
    validation_rejected = Path(OUTPUT_DIR) / "validation_rejected.jsonl"

    run_cmd(
        [
            PY, "-m", "training.trajectory_dataset.cli", "validate",
            "--input", str(mixed_path),
            "--output", str(validated_path),
            "--rejected-output", str(validation_rejected),
        ],
        "CANONICAL VALIDATION",
    )

    if count_jsonl(validation_rejected):
        raise RuntimeError("Final canonical validation rejected rows.")
else:
    print("SKIPPED: canonical validation")

## Cell 29 — Final token audit

Không crash nếu audit exit=2; set `FINAL_TOKEN_GATE_PASS`.

In [ ]:
FINAL_TOKEN_GATE_PASS = False

if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS:
    final_token_audit = Path(OUTPUT_DIR) / "final_token_audit.json"

    run_cmd(
        [
            PY, "-m", "training.trajectory_dataset.cli", "audit",
            "--input", str(validated_path),
            "--tokenizer-model-id", "Qwen/Qwen3-8B",
            "--max-seq-length", str(MAX_SEQ_LENGTH),
            "--output", str(final_token_audit),
        ],
        "FINAL MIXED TOKEN AUDIT",
        allowed_returncodes=(0, 2),
    )

    final_report = load_json(final_token_audit)
    final_danger = token_danger(final_report)
    FINAL_TOKEN_GATE_PASS = not any(final_danger.values())

    print("FINAL_TOKEN_GATE_PASS =", FINAL_TOKEN_GATE_PASS)
    print("Danger:", final_danger)
else:
    print("SKIPPED: final token audit")

## Cell 30 — Split 90/5/5

Chỉ split nếu final token gate PASS.

In [ ]:
if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS and FINAL_TOKEN_GATE_PASS:
    final_dir = Path(OUTPUT_DIR) / "final"

    run_cmd(
        [
            PY, "-m", "training.trajectory_dataset.cli", "split",
            "--input", str(validated_path),
            "--output-dir", str(final_dir),
            "--train-ratio", "0.90",
            "--val-ratio", "0.05",
            "--test-ratio", "0.05",
            "--seed", str(SEED),
        ],
        "GROUP-SAFE SPLIT",
    )

    for name in ("train", "validation", "test"):
        print(name, count_jsonl(final_dir / f"{name}.jsonl"))
else:
    print("SKIPPED: split")

## Cell 31 — Independent leakage check

Tự kiểm source_groups không overlap.

In [ ]:
if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS and FINAL_TOKEN_GATE_PASS:
    def groups_for_row(row):
        prov = row.get("provenance") or {}
        groups = prov.get("source_groups")
        if isinstance(groups, list) and groups:
            return {str(x) for x in groups if x not in {None, ""}}
        legacy = (
            prov.get("source_group")
            or prov.get("source_document_id")
            or prov.get("article_id")
            or row.get("group_id")
        )
        return {str(legacy)} if legacy not in {None, ""} else set()

    split_rows = {
        "train": read_jsonl(final_dir / "train.jsonl"),
        "validation": read_jsonl(final_dir / "validation.jsonl"),
        "test": read_jsonl(final_dir / "test.jsonl"),
    }

    group_sets = {
        split: set().union(*(groups_for_row(r) for r in data)) if data else set()
        for split, data in split_rows.items()
    }

    leaks = {}
    for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
        overlap = group_sets[a] & group_sets[b]
        if overlap:
            leaks[f"{a}<->{b}"] = sorted(overlap)[:20]

    print("Leaks:", leaks)
    if leaks:
        raise RuntimeError(f"SOURCE-GROUP LEAKAGE: {leaks}")
else:
    print("SKIPPED: leakage check")

## Cell 32 — Final report

In manifest/stats nếu build hoàn tất.

In [ ]:
if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS and FINAL_TOKEN_GATE_PASS:
    manifest = load_json(final_dir / "manifest.json")
    stats = load_json(final_dir / "dataset_stats.json")

    print("=" * 100)
    print("MANIFEST")
    print("=" * 100)
    print(json.dumps(manifest, ensure_ascii=False, indent=2))

    print("\n" + "=" * 100)
    print("STATS")
    print("=" * 100)
    print(json.dumps(stats, ensure_ascii=False, indent=2))

    print("\nFINAL DATASET BUILD: PASS")
else:
    print("FINAL REPORT SKIPPED — full gates not all passed.")

## Cell 33 — Export ZIP

Zip run folder nếu final build đã hoàn tất.

In [ ]:
if FULL_BUILD_ALLOWED and FULL_CUSTOM_GATE_PASS and FINAL_TOKEN_GATE_PASS:
    zip_base = Path(DRIVE_ROOT) / f"{RUN_NAME}_export"
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)

    print("ZIP:", zip_path)
else:
    print("ZIP export skipped.")

In [ ]:
import threading
from google.colab import runtime

DELAY_MINUTES = 3

def shutdown_runtime():
    print(f"\n⏰ Đã hết {DELAY_MINUTES} phút → ngắt Colab runtime...")
    runtime.unassign()

timer = threading.Timer(DELAY_MINUTES * 60, shutdown_runtime)
timer.daemon = True
timer.start()

print(f"✅ Đã hẹn tự ngắt runtime sau {DELAY_MINUTES} phút.")

# Config quick guide

### Khuyến nghị mặc định

```python
TOP_K = 6
AUTO_TUNE_RERANK_BATCH = True
RERANK_BATCH_SIZE = 4      # fallback; Cell 7 tự tăng theo VRAM
RERANK_BATCH_SIZE_CAP = 32

CHECKPOINT_EVERY = 100
PUBLIC_CHECKPOINT_EVERY = 100

OBSERVATION_CHAR_BUDGET = 3200
TRAJECTORY_OBSERVATION_CHAR_BUDGET = 5200
MAX_RESULT_TEXT_CHARS = 650
MAX_SEQ_LENGTH = 4096
```

### GPU

- **L4 / A10 ~24 GB:** lựa chọn khuyến nghị, auto batch 16.
- **T4 16 GB:** chạy được, auto batch 8.
- **A100 40 GB:** auto batch 24; tốt khi cần rút ngắn wall-clock time.
- **A100/H100 80 GB:** auto batch 32; thường dư cho pipeline này.

Nếu CUDA OOM: tắt `AUTO_TUNE_RERANK_BATCH` và thử `RERANK_BATCH_SIZE = 8` hoặc `4`.

Không giảm `TOP_K` hay observation budget chỉ để tăng tốc nếu mục tiêu là giữ chất lượng dataset hiện tại.

### Vì sao không tăng batch vô hạn?

`RERANK_BATCH_SIZE` chỉ batch phần reranker. Pipeline vẫn có candidate selection, E5/FAISS/BM25, sentence filtering, checkpoint và nhiều retrieval calls nối tiếp. Vì vậy batch lớn hơn VRAM cần thiết có thể chỉ tăng OOM mà không tăng tốc thêm.

**Hard stop trước Full Build** nếu có bất kỳ dấu hiệu:
- target/entity mismatch;
- compare target A/B contamination thật;
- unexpected required empty > 0;
- token supervision bị mất.

Nếu smoke sạch và đã review, mới đổi:

```python
APPROVE_FULL_BUILD = True
```
